# Mask Inventory Problem with AMPL in Python
## AMPL Modeling for Book Problem 3.3

This notebook models the mask-inventory problem from book section `3.3` with AMPL from Python using `amplpy`.


## Requirements

This notebook uses `amplpy` together with the HiGHS solver module.

If your environment does not have `amplpy`, install it first:

```python
%pip install amplpy
```

The first call to `ampl_notebook(modules=["highs"], license_uuid="default")` may download the solver module if it is not already available.


In [1]:
from __future__ import annotations

from functools import wraps
from math import isclose
from time import perf_counter


In [2]:
%pip install amplpy
try:
    from amplpy import ampl_notebook
except ImportError as exc:
    raise ImportError(
        "This notebook requires amplpy. Install it with `%pip install amplpy` before running."
    ) from exc


def timed(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = perf_counter()
        result = func(*args, **kwargs)
        elapsed = perf_counter() - start
        return result, elapsed

    return wrapper


def create_ampl_instance(solver: str = "highs"):
    ampl = ampl_notebook(modules=[solver], license_uuid="default")
    ampl.option["solver"] = solver
    ampl.option["solver_msg"] = 0
    return ampl


def round_if_close(value: float, digits: int = 4):
    rounded = round(value, digits)
    return int(round(rounded)) if isclose(rounded, round(rounded), abs_tol=1e-9) else rounded


def extract_1d_solution(variable, keys):
    raw = variable.get_values().to_dict()
    values = {}
    for key in keys:
        if key in raw:
            value = raw[key]
        else:
            value = raw[(key,)]
        values[key] = round_if_close(float(value))
    return values


Defaulting to user installation because normal site-packages is not writeable

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: /Applications/Xcode.app/Contents/Developer/usr/bin/python3 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Problem: Four-Month Production Plan

The model minimizes extra-production and inventory costs while meeting the forecast demand over four months, starting from zero inventory and ending with zero inventory.


In [3]:
MONTHS = [1, 2, 3, 4]
DEMAND = {1: 2800, 2: 2200, 3: 3200, 4: 2500}
NORMAL_CAPACITY = 2700
EXTRA_CAPACITY = 300
EXTRA_COST = 10
INVENTORY_COST = 2
EXPECTED = {
    "normal": {1: 2700, 2: 2700, 3: 2700, 4: 2500},
    "extra": {1: 100, 2: 0, 3: 0, 4: 0},
    "inventory": {0: 0, 1: 0, 2: 500, 3: 0, 4: 0},
    "cost": 2_000,
}


@timed
def solve_mask_inventory_with_ampl(solver: str = "highs"):
    ampl = create_ampl_instance(solver)
    ampl.eval(
        r'''
        set MONTHS ordered;
        set MONTHS_WITH_ZERO ordered;

        param demand {MONTHS} >= 0;
        param normal_capacity >= 0;
        param extra_capacity >= 0;
        param extra_cost >= 0;
        param inventory_cost >= 0;

        var NormalProd {t in MONTHS} integer >= 0;
        var ExtraProd {t in MONTHS} integer >= 0;
        var Inventory {t in MONTHS_WITH_ZERO} integer >= 0;

        minimize TotalCost:
            sum {t in MONTHS} extra_cost * ExtraProd[t]
            + sum {t in MONTHS} inventory_cost * Inventory[t];

        subject to NormalCapacity {t in MONTHS}:
            NormalProd[t] <= normal_capacity;

        subject to ExtraCapacity {t in MONTHS}:
            ExtraProd[t] <= extra_capacity;

        subject to Balance {t in MONTHS}:
            Inventory[t - 1] + NormalProd[t] + ExtraProd[t] = demand[t] + Inventory[t];

        subject to InitialInventory:
            Inventory[0] = 0;

        subject to FinalInventory:
            Inventory[4] = 0;
        '''
    )
    ampl.set["MONTHS"] = MONTHS
    ampl.set["MONTHS_WITH_ZERO"] = [0, 1, 2, 3, 4]
    ampl.param["demand"] = DEMAND
    ampl.param["normal_capacity"] = NORMAL_CAPACITY
    ampl.param["extra_capacity"] = EXTRA_CAPACITY
    ampl.param["extra_cost"] = EXTRA_COST
    ampl.param["inventory_cost"] = INVENTORY_COST
    ampl.solve()

    return {
        "normal": extract_1d_solution(ampl.var["NormalProd"], MONTHS),
        "extra": extract_1d_solution(ampl.var["ExtraProd"], MONTHS),
        "inventory": extract_1d_solution(ampl.var["Inventory"], [0, 1, 2, 3, 4]),
        "cost": round_if_close(ampl.obj["TotalCost"].value()),
    }


ampl_result, ampl_time = solve_mask_inventory_with_ampl()
print("=== MASK INVENTORY RESULTS WITH AMPL ===")
print(f"Solution -> {ampl_result}")
print(f"Time     -> {ampl_time:.8f} seconds")

assert ampl_result == EXPECTED
print("AMPL matches the published monthly production plan.")


/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


AMPL Version 20250901 (Darwin-22.6.0, 64-bit)
Demo license with maintenance expiring 20270131.
Using license file "/Users/lfuentesl/Library/Python/3.9/lib/python/site-packages/ampl_module_base/bin/ampl.lic".

HiGHS 1.11.0: === MASK INVENTORY RESULTS WITH AMPL ===
Solution -> {'normal': {1: 2700, 2: 2700, 3: 2700, 4: 2500}, 'extra': {1: 100, 2: 0, 3: 0, 4: 0}, 'inventory': {0: 0, 1: 0, 2: 500, 3: 0, 4: 0}, 'cost': 2000}
Time     -> 1.65643275 seconds
AMPL matches the published monthly production plan.


## Variant Note

Section `3.3` proposes a student variation with a second article. This notebook closes with the printed one-product base model only.
